<a href="https://colab.research.google.com/github/kalpesh5536/Twitter-airline-sentiment-analysis-using-NLP-pipleline-and-Ml-models/blob/main/Twitter_airline_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

𝗡𝗟𝗣 𝗣𝗿𝗼𝗷𝗲𝗰𝘁: Twitter airline sentiment analysis



In [2]:
# Basic
import pandas as pd
import numpy as np

# Text cleaning
import re
import nltk

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Download stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
# Upload Tweets.csv in Colab first

df = pd.read_csv("Tweets.csv")

# Keep only needed columns
df = df[['airline_sentiment', 'text']]

# Rename for simplicity
df.columns = ['sentiment', 'text']

# Remove missing values
df = df.dropna()

df.head()

,sentiment,text
0,neutral,@VirginAmerica What @dhepburn said.
1,positive,@VirginAmerica plus you've added commercials t...
2,neutral,@VirginAmerica I didn't today... Must mean I n...
3,negative,@VirginAmerica it's really aggressive to blast...
4,negative,@VirginAmerica and it's a really big bad thing...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  14640 non-null  object
 1   text       14640 non-null  object
dtypes: object(2)
memory usage: 228.9+ KB


In [4]:
# Data Understanding
print("Total Rows:", len(df))

print("\nClass Distribution:")
print(df['sentiment'].value_counts())

print("\nSample Tweet:")
print(df['text'].iloc[0])

Total Rows: 14640

Class Distribution:
sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64

Sample Tweet:
@VirginAmerica What @dhepburn said.


In [6]:
# Preprocessing (Cleaning Text)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()  # lowercase

    text = re.sub(r'http\S+|www\S+', '', text)  # remove links
    text = re.sub(r'[^a-z\s]', '', text)        # remove symbols

    words = text.split()

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    # stemming
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

# Apply cleaning
df['clean_text'] = df['text'].apply(clean_text)

df[['text', 'clean_text']].head()

,text,clean_text
0,@VirginAmerica What @dhepburn said.,virginamerica dhepburn said
1,@VirginAmerica plus you've added commercials t...,virginamerica plu youv ad commerci experi tacki
2,@VirginAmerica I didn't today... Must mean I n...,virginamerica didnt today must mean need take ...
3,@VirginAmerica it's really aggressive to blast...,virginamerica realli aggress blast obnoxi ente...
4,@VirginAmerica and it's a really big bad thing...,virginamerica realli big bad thing


In [7]:
# Convert Labels to Numbers
le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])

print("Label Mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Label Mapping: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


In [8]:
# Feature Engineering
# 1) Bag of words
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

In [9]:
# 2) TF-IDF
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_text'])
y = df['sentiment']

In [10]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
# Train Models
# Logistic Regression

lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)


In [12]:
# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [13]:
# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)


In [14]:
# Evaluation
def evaluate(y_test, y_pred):
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall   :", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))
    print("-" * 40)

print("Logistic Regression")
evaluate(y_test, y_pred_lr)

print("Naive Bayes")
evaluate(y_test, y_pred_nb)

print("Decision Tree")
evaluate(y_test, y_pred_dt)

Logistic Regression
Accuracy : 0.7954234972677595
Precision: 0.7855904855246059
Recall   : 0.7954234972677595
F1 Score : 0.7833121177824094
----------------------------------------
Naive Bayes
Accuracy : 0.7353142076502732
Precision: 0.7492834895468619
Recall   : 0.7353142076502732
F1 Score : 0.6867183263293224
----------------------------------------
Decision Tree
Accuracy : 0.6868169398907104
Precision: 0.6914035331220378
Recall   : 0.6868169398907104
F1 Score : 0.6889009012227856
----------------------------------------


Pipeline Flow:
Raw Data → Preprocessing → Feature Engineering → Model Training → Evaluation

Conclusion:
After comparing all models, the best model is the one with highest F1 score.

Logistic Regression gives stable performance.
Naive Bayes is fast but slightly less accurate.
Decision Tree may give high accuracy but can overfit.

TF-IDF performed better than Bag of Words because it focuses on important words.

Preprocessing improved performance by removing noise and unnecessary words.